# Ridge Regression Code Companion

This notebook connects Ridge Regression code with the theory: Linear Regression plus an L2 penalty that shrinks coefficients and helps reduce overfitting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Load a Regression Dataset

Ridge Regression predicts a continuous target, just like Linear Regression. The difference is that Ridge controls large coefficients using regularization.


In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print("Rows and features:", X.shape)
X.head()


## Split the Data

Training data teaches the model. Testing data checks whether the model generalizes.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## Train Linear Regression and Ridge Regression

Scaling is important because Ridge penalizes coefficients. Features should be on comparable scales before applying the L2 penalty.


In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0))
])

linear_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)


## Compare Performance

Ridge may slightly increase training error, but it often improves stability by shrinking coefficients.


In [ ]:
def regression_metrics(name, model):
    y_pred = model.predict(X_test)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    }

pd.DataFrame([
    regression_metrics("Linear Regression", linear_model),
    regression_metrics("Ridge Regression", ridge_model)
])


## Compare Coefficients

Ridge uses the L2 penalty to keep coefficients smaller. Smaller coefficients usually mean the model is less sensitive to small changes in the data.


In [ ]:
linear_coef = linear_model.named_steps["model"].coef_
ridge_coef = ridge_model.named_steps["model"].coef_

coef_table = pd.DataFrame({
    "feature": X.columns,
    "linear_coefficient": linear_coef,
    "ridge_coefficient": ridge_coef,
    "absolute_shrinkage": np.abs(linear_coef) - np.abs(ridge_coef)
}).sort_values("absolute_shrinkage", ascending=False)

coef_table


## Effect of Alpha

`alpha` is the code name for $\lambda$. Larger values create stronger regularization and usually smaller coefficients.


In [ ]:
rows = []
for alpha in [0.01, 0.1, 1, 10, 100, 1000]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    coefficients = model.named_steps["model"].coef_
    rows.append({
        "alpha": alpha,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
        "coefficient_norm": np.linalg.norm(coefficients)
    })

pd.DataFrame(rows)
